In [0]:
from pyspark.sql.functions import lit

result = []
for table in spark.catalog.listTables("bharatmart.gold"):
    df = spark.table(f"bharatmart.gold.{table.name}")
    for field in df.schema.fields:
        result.append((table.name, field.name, str(field.dataType)))

schema_df = spark.createDataFrame(result, ["table_name", "column_name", "data_type"])
display(schema_df.orderBy("table_name", "column_name"))

In [0]:
import os

# ── config ──────────────────────────────────────────────────
EXPORT_PATH = "/dbfs/tmp/bharatmart_kaggle_export/"
ADLS_TMP    = "abfss://bharatmart@stbharatmartdev.dfs.core.windows.net/kaggle_export/"

# only dim_ and fact_ tables — skip agg_
TABLES = [
    # dimensions
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_warehouse",
    "dim_geo",
    "dim_date",
    # facts
    "fact_orders",
    "fact_payments",
    "fact_shipments",
    "fact_reviews",
    "fact_refunds",
    "fact_commissions",
    "fact_inventory",
    "fact_sessions",
    "fact_cart_events",
    "fact_campaign_responses",
]

print(f"Exporting {len(TABLES)} tables...\n")

for tbl in TABLES:
    print(f"  exporting {tbl}...", end=" ")
    df = spark.table(f"bharatmart.gold.{tbl}")
    
    # write to ADLS as single CSV file
    (df.coalesce(1)
       .write
       .mode("overwrite")
       .option("header", "true")
       .csv(f"{ADLS_TMP}{tbl}/")
    )
    
    # rename the part file to a clean name
    files = dbutils.fs.ls(f"{ADLS_TMP}{tbl}/")
    part_file = [f.path for f in files if f.name.startswith("part-")][0]
    dbutils.fs.cp(part_file, f"{ADLS_TMP}{tbl}.csv")
    dbutils.fs.rm(f"{ADLS_TMP}{tbl}/", True)
    
    count = df.count()
    print(f"done — {count:,} rows → {tbl}.csv")

print("\n=== ALL EXPORTS COMPLETE ===")
print(f"Files at: {ADLS_TMP}")
print("\nFiles ready:")
for f in dbutils.fs.ls(ADLS_TMP):
    size_mb = f.size / (1024*1024)
    print(f"  {f.name:<45} {size_mb:.1f} MB")

In [0]:
spark.table("bharatmart.gold.fact_commissions").printSchema()
spark.table("bharatmart.gold.fact_commissions").limit(3).display()